# Tournament

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

In [ ]:
print("Compiling full 2019 2500-tick bars...")
bars_list = []

for m in range(1, 13):
    month_str = f"2019{m:02d}"
    print(f"Loading {month_str}...")
    
    # 1. Load Parquet
    bid = pd.read_parquet(f"../data/processed/eurusd_dukascopy_bid_{month_str}.parquet")
    ask = pd.read_parquet(f"../data/processed/eurusd_dukascopy_ask_{month_str}.parquet")
    
    # 2. Sort (required for merge_asof)
    bid['datetime'] = pd.to_datetime(bid['datetime'])
    ask['datetime'] = pd.to_datetime(ask['datetime'])
    bid = bid.sort_values('datetime')
    ask = ask.sort_values('datetime')
    
    # 3. Merge and Mid-Price
    merged = pd.merge_asof(
        bid, 
        ask[['datetime', 'price']], 
        on='datetime', 
        direction='backward', 
        suffixes=('_bid', '_ask')
    )
    merged['mid_price'] = (merged['price_bid'] + merged['price_ask']) / 2
    
    # 4. Sample immediately to save memory (2500 ticks)
    sampled = merged.iloc[::2500].copy()
    bars_list.append(sampled)

# 5. Combine the whole year into df_bars
df_bars = pd.concat(bars_list).reset_index(drop=True)
df_bars.set_index('datetime', inplace=True)

# 6. Calculate log returns for the whole continuous year
df_bars['log_returns'] = np.log(df_bars['mid_price'] / df_bars['mid_price'].shift(1))
df_bars.dropna(inplace=True)

print(f"\nSuccess! Total 2019 bars ready: {len(df_bars)}")

In [ ]:
# ==========================================
# 1. PREPARE THE TARGET VARIABLE
# ==========================================
# Assuming df_bars is your full 2019 dataframe of 2500-tick bars
df_bars['vol_proxy'] = np.abs(df_bars['log_returns']) * 1e6
df_bars['month'] = df_bars.index.month
df_bars.dropna(inplace=True)

results = []
eval_data_list = [] # Stockpile out-of-sample predictions for the dashboard

print("Starting Walk-Forward Volatility Tournament...")

# ==========================================
# 2. THE WALK-FORWARD TOURNAMENT
# ==========================================
for m in range(1, 10): # Loop from Jan to Sep (so last test is Dec)
    train_months = [m, m+1, m+2]
    test_month = m + 3
    
    # Split the data
    train_data = df_bars[df_bars['month'].isin(train_months)]['vol_proxy']
    test_data = df_bars[df_bars['month'] == test_month]['vol_proxy']
    test_raw_returns = df_bars[df_bars['month'] == test_month]['log_returns']
    
    if len(test_data) == 0:
        continue
        
    print(f"\nTraining on Months {train_months} | Testing on Month {test_month}")
    
    # To do 1-step ahead forecasting, we need the combined series
    full_data = pd.concat([train_data, test_data])
    start_idx = len(train_data)
    end_idx = len(full_data) - 1

    # --- MODEL 1: The Baseline (SMA of last 5 bars) ---
    sma_forecast = full_data.rolling(window=5).mean().shift(1).iloc[start_idx:]
    sma_rmse = np.sqrt(mean_squared_error(test_data, sma_forecast.fillna(method='bfill')))

    # --- MODEL 2: AR(1) ---
    print("  Fitting AR(1)...")
    ar_model = AutoReg(train_data, lags=1).fit()
    ar_forecast = ar_model.predict(start=start_idx, end=end_idx, dynamic=False)
    ar_rmse = np.sqrt(mean_squared_error(test_data, ar_forecast))

    # --- MODEL 3: Markov-Switching AR(1) (The Filtered Hack) ---
    print("  Fitting MS-AR(1)...")
    try:
        # Feed the entire 4-month block (Train + Test)
        ms_model = sm.tsa.MarkovAutoregression(
            endog=full_data, 
            k_regimes=2, 
            order=1, 
            switching_ar=False, 
            switching_variance=True
        )
        
        # 'bfgs' is more stable, 'disp=False' keeps it quiet
        ms_res = ms_model.fit(method='bfgs', disp=False)
        
        # THE FIX: Grab strictly the last N elements to match test_data perfectly
        ms_forecast = ms_res.fittedvalues[-len(test_data):]
        ms_rmse = np.sqrt(mean_squared_error(test_data, ms_forecast))
        
        # Store for final dashboard evaluation
        eval_data_list.append(pd.DataFrame({
            'raw_returns': test_raw_returns.values,
            'actual_vol': test_data.values,
            'pred_vol': ms_forecast.values
        }))
        
    except Exception as e:
        print(f"  MS-AR(1) Failed: {repr(e)}")
        ms_rmse = np.nan

    # Record Results
    results.append({
        'Test Month': test_month,
        'SMA RMSE': round(sma_rmse, 2),
        'AR(1) RMSE': round(ar_rmse, 2),
        'MS-AR(1) RMSE': round(ms_rmse, 2)
    })

# Print the Final Scoreboard
results_df = pd.DataFrame(results)
print("\n=== FINAL TOURNAMENT RESULTS (RMSE) ===")
print("Lower is better. RMSE represents the average error in scaled volatility units.")
print(results_df.to_string(index=False))

# ==========================================
# 3. THE EVALUATION DASHBOARD
# ==========================================
def calculate_metrics_and_plot(df_eval, alpha=0.05, target_vol_bps=100):
    y_true = df_eval['actual_vol'].values
    y_pred = df_eval['pred_vol'].values
    returns = df_eval['raw_returns'].values
    
    # --- 1. Calculate Prediction Intervals (95%) ---
    residuals = y_true - y_pred
    rolling_std = pd.Series(residuals).rolling(window=20, min_periods=1).std().fillna(method='bfill').values
    z_score = stats.norm.ppf(1 - alpha/2)
    
    upper_bound = y_pred + (z_score * rolling_std)
    lower_bound = np.maximum(0, y_pred - (z_score * rolling_std))
    
    # --- 2. STATISTICAL METRICS ---
    stats_metrics = {}
    stats_metrics['RMSE'] = np.sqrt(np.mean(residuals**2))
    stats_metrics['MAE'] = np.mean(np.abs(residuals))
    
    safe_y_true = np.where(y_true == 0, 1e-6, y_true)
    stats_metrics['MAPE (%)'] = np.mean(np.abs(residuals / safe_y_true)) * 100
    
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    stats_metrics['R-Squared'] = 1 - (ss_res / ss_tot)
    
    in_bounds = (y_true >= lower_bound) & (y_true <= upper_bound)
    stats_metrics['Coverage (PICP) (%)'] = np.mean(in_bounds) * 100
    
    width = upper_bound - lower_bound
    penalty_low = np.where(y_true < lower_bound, (2/alpha) * (lower_bound - y_true), 0)
    penalty_high = np.where(y_true > upper_bound, (2/alpha) * (y_true - upper_bound), 0)
    stats_metrics['Winkler Score'] = np.mean(width + penalty_low + penalty_high)
    
    actual_diff = np.diff(y_true)
    pred_diff = np.diff(y_pred)
    stats_metrics['Dir. Accuracy (MDA) (%)'] = np.mean(np.sign(actual_diff) == np.sign(pred_diff)) * 100
    stats_metrics['Bias'] = np.mean(residuals)
    stats_metrics['Correlation'] = stats.pearsonr(y_true, y_pred)[0]
    
    naive_err = np.sum((y_true[1:] - y_true[:-1])**2)
    stats_metrics['Theils U'] = np.sqrt(ss_res) / (np.sqrt(naive_err) + 1e-6)

    # --- 3. FINANCIAL METRICS ---
    fin_metrics = {}
    position_size = np.clip(target_vol_bps / (y_pred + 1e-6), 0, 5)
    strat_returns = returns * position_size
    ann_factor = 3500 
    
    fin_metrics['Total Return (%)'] = (np.exp(np.sum(strat_returns)) - 1) * 100
    fin_metrics['Ann. Return (%)'] = np.mean(strat_returns) * ann_factor * 100
    fin_metrics['Ann. Volatility (%)'] = np.std(strat_returns) * np.sqrt(ann_factor) * 100
    fin_metrics['Sharpe Ratio'] = fin_metrics['Ann. Return (%)'] / (fin_metrics['Ann. Volatility (%)'] + 1e-6)
    
    downside_returns = strat_returns[strat_returns < 0]
    downside_vol = np.std(downside_returns) * np.sqrt(ann_factor) * 100
    fin_metrics['Sortino Ratio'] = fin_metrics['Ann. Return (%)'] / (downside_vol + 1e-6)
    
    cum_returns = np.cumsum(strat_returns)
    running_max = np.maximum.accumulate(cum_returns)
    drawdowns = running_max - cum_returns
    fin_metrics['Max Drawdown (%)'] = np.max(drawdowns) * 100
    fin_metrics['Calmar Ratio'] = fin_metrics['Ann. Return (%)'] / (fin_metrics['Max Drawdown (%)'] + 1e-6)
    fin_metrics['Hit Rate (%)'] = np.mean(strat_returns > 0) * 100
    fin_metrics['95% VaR (bps)'] = abs(np.percentile(strat_returns, 5)) * 10000
    fin_metrics['CVaR (bps)'] = abs(np.mean(strat_returns[strat_returns <= np.percentile(strat_returns, 5)])) * 10000

    # --- 4. PRINT TABLES ---
    print("\n" + "="*40)
    print("📈 STATISTICAL FORECAST METRICS (FULL OOS)")
    print("="*40)
    for k, v in stats_metrics.items():
        print(f"{k:<25}: {v:.4f}")

    print("\n" + "="*40)
    print("💰 FINANCIAL STRATEGY METRICS (Vol-Targeted)")
    print("="*40)
    for k, v in fin_metrics.items():
        print(f"{k:<25}: {v:.4f}")

    # --- 5. PLOTTING DASHBOARD ---
    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
    
    axes[0].plot(y_true, label='Actual Volatility', color='black', alpha=0.5, linewidth=1)
    axes[0].plot(y_pred, label='MS-AR Predicted Vol', color='tab:red', alpha=0.9, linewidth=1.5)
    axes[0].set_title('Out-of-Sample Volatility Forecast (MS-AR)', fontweight='bold')
    axes[0].legend(loc='upper right')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(y_true, color='black', alpha=0.5, linewidth=1)
    axes[1].fill_between(range(len(y_pred)), lower_bound, upper_bound, color='tab:blue', alpha=0.3, label='95% Prediction Interval')
    axes[1].plot(y_pred, color='tab:blue', linestyle='--', linewidth=1, label='Point Forecast')
    axes[1].set_title(f'Uncertainty Bounds (Winkler Score: {stats_metrics["Winkler Score"]:.2f} | Coverage: {stats_metrics["Coverage (PICP) (%)"]:.1f}%)', fontweight='bold')
    axes[1].legend(loc='upper right')
    axes[1].grid(True, alpha=0.3)
    
    raw_cum_pnl = np.cumsum(returns)
    strat_cum_pnl = np.cumsum(strat_returns)
    axes[2].plot(raw_cum_pnl, label='Raw Market Returns (Buy & Hold)', color='gray', linestyle='--')
    axes[2].plot(strat_cum_pnl, label='Vol-Targeted Strategy Returns', color='tab:green', linewidth=2)
    axes[2].set_title(f'Strategy PnL (Sharpe: {fin_metrics["Sharpe Ratio"]:.2f} | Max DD: {fin_metrics["Max Drawdown (%)"]:.1f}%)', fontweight='bold')
    axes[2].legend(loc='upper left')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Execute the dashboard if we have valid OOS data
if eval_data_list:
    df_eval_master = pd.concat(eval_data_list, ignore_index=True)
    print("\nBuilding Full Evaluation Dashboard...")
    calculate_metrics_and_plot(df_eval_master)